In [ ]:
import os
import string
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences

def clean_text(text):
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return text.strip()

BASE_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
MODEL_DIR = os.path.join(BASE_DIR, "model") if os.path.exists(os.path.join(BASE_DIR, "model")) else BASE_DIR

# Load model and class maps
model = tf.keras.models.load_model(os.path.join(MODEL_DIR, "rnn_gk_model.keras"))
q_word_index = np.load(os.path.join(MODEL_DIR, "question_word_index.npy"), allow_pickle=True).item()
id_to_answer = np.load(os.path.join(MODEL_DIR, "id_to_answer.npy"), allow_pickle=True).item()
max_length_q = int(np.load(os.path.join(MODEL_DIR, "max_length.npy")))

while True:
    user_input = input("\nYou: ")
    if user_input.lower() == "exit":
        break

    cleaned = clean_text(user_input)
    words = cleaned.split()
    seq = [q_word_index.get(w, q_word_index.get("<OOV>", 1)) for w in words]
    padded = pad_sequences([seq], maxlen=max_length_q, padding="post")

    prediction = model.predict(padded, verbose=0)
    predicted_class = np.argmax(prediction[0])
    
    # Get answer text from predicted class ID
    answer = id_to_answer.get(predicted_class, "I don't know")
    print("RNN:", answer)